In [487]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pymap3d as pm
import math
import seaborn as sns
import datetime as dt
import time
import os
import glob
from pathlib import Path

# ========================
# КОНФИГУРАЦИЯ
# ========================
# SOLUTIONS_DIR = "solutions"  # Папка с вашими POS-файлами
SOLUTIONS_DIR = "solutions_gps"  # Папка с вашими POS-файлами
# SOLUTIONS_DIR = "solutions_nophase"  # Папка с вашими POS-файлами
# SOLUTIONS_DIR = "solutions_nophase_gps"  # Папка с вашими POS-файлами
# SOLUTIONS_DIR = "solutions_nophase_rel"  # Папка с вашими POS-файлами
# SOLUTIONS_DIR = "solutions_nophase_gps_rel" 
# SOLUTIONS_DIR = "solutions_all_rel" 
# SOLUTIONS_DIR = "solutions_nophase_all_rel" 

# SOLUTIONS_DIR = "solutions_gps_rel"  # Папка с вашими POS-файлами
# ========================
# КОНФИГУРАЦИЯ
# ========================
REF_POINT = {'x': 54.9872753361111, 'y': 82.864814275, 'z': 109.647}  # Эталонные координаты
SOL = 10

# ========================
# ФИЛЬТРЫ ПО ОШИБКАМ ENU
# ========================
MAX_HEIGHT_ERROR = .3      # Максимальное допустимое отклонение высоты в метрах
MAX_EAST_ERROR = .2      # Максимальное допустимое отклонение по востоку (E)
MAX_NORTH_ERROR = .2       # Максимальное допустимое отклонение по северу (N)
MAX_HORIZONTAL_ERROR = 15  # Максимальное допустимое горизонтальное отклонение (2D)
MAX_3D_ERROR = 20.0        # Максимальное допустимое 3D отклонение
IQR_FACTOR = 1.5 
# ========================
# СМЕЩЕНИЕ (BIAS) ДЛЯ КООРДИНАТ ENU — ПРИМЕНЯЕТСЯ ПЕРЕД ВЫЧИСЛЕНИЕМ СТАТИСТИК
# ========================
# ENU_BIAS_E = 0.0    # Смещение по востоку (м) — например, +0.03
# ENU_BIAS_N = 0.1055    # Смещение по северу (м) — например, -0.01
# ENU_BIAS_U = 0.4187    # Смещение по высоте (м) — например, +0.05

# ENU_BIAS_E = 0.027    # Смещение по востоку (м) — например, +0.03
# ENU_BIAS_N = 0.0925    # Смещение по северу (м) — например, -0.01
# ENU_BIAS_U = 0.3607  

# ENU_BIAS_E = .027    # Смещение по востоку (м) — например, +0.03
# ENU_BIAS_N = .013    # Смещение по северу (м) — например, -0.01
# ENU_BIAS_U = .058  

ENU_BIAS_E = 0    # Смещение по востоку (м) — например, +0.03
ENU_BIAS_N = 0    # Смещение по северу (м) — например, -0.01
ENU_BIAS_U = 0  
# Фильтр по типу решений
ONLY_FIXED_SOLUTIONS = False  # Если True - использовать только фиксированные решения (quality=1)
# ========================
# ФУНКЦИИ ПРЕОБРАЗОВАНИЯ
# ========================

def reduction_to_enu(df, ref_point):
    """Преобразует координаты в систему ENU относительно эталона"""
    if len(df) == 0:
        return df
    
    enu = []
    for i in range(len(df)):
        try:
            e, n, u = pm.geodetic2enu(
                df['latitude(deg)'].iloc[i], 
                df['longitude(deg)'].iloc[i], 
                df['height(m)'].iloc[i], 
                ref_point['x'], ref_point['y'], ref_point['z']
            )
            enu.append([e, n, u])
        except Exception as e:
            enu.append([np.nan, np.nan, np.nan])
    
    enu_df = pd.DataFrame(enu, columns=['E', 'N', 'U'])
    result = pd.concat([df.reset_index(drop=True), enu_df], axis=1)
    
    # Удаляем строки с NaN в ENU координатах
    initial_count = len(result)
    result = result.dropna(subset=['E', 'N', 'U'])
    final_count = len(result)
    
    if initial_count != final_count:
        print(f"   Удалено {initial_count - final_count} строк с ошибками преобразования")
    
    return result

def filter_fixed_solutions(df_enu):
    """
    Фильтрует только фиксированные решения (quality = 1)
    """
    if len(df_enu) == 0:
        return df_enu
    
    if 'quality' not in df_enu.columns:
        return df_enu
    
    # Фильтруем только фиксированные решения
    df_fixed = df_enu[df_enu['quality'] == 1].copy()
    
    initial_count = len(df_enu)
    final_count = len(df_fixed)
    
    if initial_count > 0:
        print(f"   Фикс: {initial_count} → {final_count} решений (нефикс: {initial_count - final_count})")
    
    return df_fixed

def filter_solutions_by_enu_limits(df_enu, 
                                 max_east=MAX_EAST_ERROR,
                                 max_north=MAX_NORTH_ERROR, 
                                 max_height=MAX_HEIGHT_ERROR,
                                 max_horizontal=MAX_HORIZONTAL_ERROR,
                                 max_3d=MAX_3D_ERROR):
    """
    Фильтрует решения по всем заданным пределам ENU
    """
    if len(df_enu) == 0:
        return df_enu
    
    df_filtered = df_enu.copy()
    
    # Применяем фильтры последовательно
    filters_applied = []
    
    # Фильтр по востоку (E)
    if max_east is not None and 'E' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['E'].abs() <= max_east]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"E≤{max_east}м(-{filtered_count})")
    
    # Фильтр по северу (N)
    if max_north is not None and 'N' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['N'].abs() <= max_north]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"N≤{max_north}м(-{filtered_count})")
    
    # Фильтр по высоте (U)
    if max_height is not None and 'U' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['U'].abs() <= max_height]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"U≤{max_height}м(-{filtered_count})")
    
    # Фильтр по горизонтальной ошибке (2D)
    if max_horizontal is not None and all(col in df_filtered.columns for col in ['E', 'N']):
        initial_count = len(df_filtered)
        horizontal_errors = np.sqrt(df_filtered['E']**2 + df_filtered['N']**2)
        df_filtered = df_filtered[horizontal_errors <= max_horizontal]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"2D≤{max_horizontal}м(-{filtered_count})")
    
    # Фильтр по 3D ошибке
    if max_3d is not None and all(col in df_filtered.columns for col in ['E', 'N', 'U']):
        initial_count = len(df_filtered)
        errors_3d = np.sqrt(df_filtered['E']**2 + df_filtered['N']**2 + df_filtered['U']**2)
        df_filtered = df_filtered[errors_3d <= max_3d]
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"3D≤{max_3d}м(-{filtered_count})")
    
    # Выводим информацию о примененных фильтрах
    if filters_applied:
        print(f"   Фильтры: {', '.join(filters_applied)}")
    
    initial_total = len(df_enu)
    final_total = len(df_filtered)
    
    if initial_total > 0:
        print(f"   Итог: {initial_total} → {final_total} решений")
    
    return df_filtered

def filter_outliers_by_iqr(df_enu, iqr_factor=1.5, components=['E', 'N', 'U']):
    """
    Фильтрует выбросы по методу IQR для указанных компонентов E, N, U.
    Удаляет строки, где хотя бы один компонент выходит за пределы [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
    
    Параметры:
        df_enu: DataFrame с колонками E, N, U
        iqr_factor: коэффициент для IQR (по умолчанию 1.5 — стандартный)
        components: список компонентов для фильтрации
    
    Возвращает:
        Отфильтрованный DataFrame
    """
    if len(df_enu) == 0:
        return df_enu
    
    df_filtered = df_enu.copy()
    initial_count = len(df_filtered)
    filters_applied = []
    
    for comp in components:
        if comp not in df_filtered.columns:
            continue
            
        Q1 = df_filtered[comp].quantile(0.25)
        Q3 = df_filtered[comp].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - iqr_factor * IQR
        upper_bound = Q3 + iqr_factor * IQR
        
        # Сохраняем только строки, где значение в пределах
        mask = (df_filtered[comp] >= lower_bound) & (df_filtered[comp] <= upper_bound)
        df_filtered = df_filtered[mask]
        
        filtered_count = initial_count - len(df_filtered)
        if filtered_count > 0:
            filters_applied.append(f"{comp}-IQR(-{filtered_count})")
    
    if filters_applied:
        print(f"   🚫 Выбросы: {', '.join(filters_applied)}")
        print(f"   Итог: {initial_count} → {len(df_filtered)} решений после IQR")
    
    return df_filtered

def select_best_solutions(df_enu, sol_count=SOL):
    """
    Выбирает лучшие SOL решений по близости к эталону с учетом всех фильтров
    """
    if len(df_enu) == 0:
        return df_enu
    
    # Сначала применяем фильтр по типу решений (фикс/нефикс)
    if ONLY_FIXED_SOLUTIONS:
        df_filtered = filter_fixed_solutions(df_enu)
    else:
        df_filtered = df_enu.copy()
    
    if len(df_filtered) == 0:
        print("   ❌ Нет решений после фильтрации по типу")
        return df_filtered
    
    # Затем применяем фильтры по пределам ENU
    df_filtered = filter_solutions_by_enu_limits(
        df_filtered,
        max_east=MAX_EAST_ERROR,
        max_north=MAX_NORTH_ERROR,
        max_height=MAX_HEIGHT_ERROR,
        max_horizontal=MAX_HORIZONTAL_ERROR,
        max_3d=MAX_3D_ERROR
    )
    
    if len(df_filtered) == 0:
        print("   ❌ Нет решений после фильтрации по ENU пределам")
        return df_filtered
    
    # ✅ НОВЫЙ ШАГ: Фильтрация выбросов по IQR
    df_filtered = filter_outliers_by_iqr(df_filtered, iqr_factor=1.5, components=['E', 'N', 'U'])
    
    if len(df_filtered) == 0:
        print("   ❌ Нет решений после фильтрации выбросов (IQR)")
        return df_filtered
    
    # Вычисляем 3D расстояние для отфильтрованных решений
    df_filtered = df_filtered.copy()
    df_filtered['distance_3d'] = np.sqrt(
        df_filtered['E']**2 + 
        df_filtered['N']**2 + 
        df_filtered['U']**2
    )
    
    # Берем самые близкие решения из отфильтрованных
    if len(df_filtered) <= sol_count:
        best_solutions = df_filtered
    else:
        best_solutions = df_filtered.nsmallest(sol_count, 'distance_3d')
    
    solution_type = "ФИКСИРОВАННЫХ" if ONLY_FIXED_SOLUTIONS else "всех"
    print(f"   ✅ Отобрано {len(best_solutions)} {solution_type} решений")
    
    return best_solutions

# ========================
# ФУНКЦИИ ЧТЕНИЯ ФАЙЛОВ
# ========================

def read_pos_file_corrected(file_path):
    """Упрощенное и корректное чтение POS-файлов"""
    try:
        # Читаем файл, пропуская комментарии
        df = pd.read_csv(file_path, 
                        sep=r'\s+',
                        comment='%',
                        header=None,
                        names=['date', 'time', 'lat', 'lon', 'height', 'Q', 'ns', 
                               'sde', 'sdn', 'sdu', 'sdne', 'sdeu', 'sdun', 'age', 'ratio'],
                        engine='python',
                        skipinitialspace=True,
                        skiprows=1)
        
        # Создаем временную метку
        try:
            df['UTC'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str))
        except:
            df['UTC'] = pd.date_range(start='2024-01-01', periods=len(df), freq='1S')[:len(df)]
        
        # Стандартизируем имена колонок
        df = df.rename(columns={
            'lat': 'latitude(deg)',
            'lon': 'longitude(deg)', 
            'height': 'height(m)',
            'Q': 'quality'
        })
        
        # Анализ качества решений
        total_solutions = len(df)
        fixed_solutions = len(df[df['quality'] == 1])
        fix_percentage = (fixed_solutions / total_solutions * 100) if total_solutions > 0 else 0
        
        print(f"✅ Загружено: {total_solutions} записей (фикс: {fixed_solutions}, {fix_percentage:.1f}%)")
        return df
        
    except Exception as e:
        print(f"❌ Ошибка чтения {file_path}: {e}")
        return pd.DataFrame()

# ========================
# ФУНКЦИИ АНАЛИЗА
# ========================

def calculate_comprehensive_statistics(df, segment_name=""):
    """
    Комплексный расчет статистики для выбранных решений
    Применяет смещение ENU перед вычислением статистик
    """
    if len(df) == 0:
        return None
    
    # Создаем копию для анализа
    df_analysis = df.copy()
    
    # Применяем смещение ENU (если заданы)
    bias_applied = False
    if ENU_BIAS_E != 0.0 and 'E' in df_analysis.columns:
        df_analysis['E'] = df_analysis['E'] - ENU_BIAS_E
        bias_applied = True
    if ENU_BIAS_N != 0.0 and 'N' in df_analysis.columns:
        df_analysis['N'] = df_analysis['N'] - ENU_BIAS_N
        bias_applied = True
    if ENU_BIAS_U != 0.0 and 'U' in df_analysis.columns:
        df_analysis['U'] = df_analysis['U'] - ENU_BIAS_U
        bias_applied = True
    
    if bias_applied:
        print(f"   ⚙️  Применено смещение ENU: E={ENU_BIAS_E:+.4f}m, N={ENU_BIAS_N:+.4f}m, U={ENU_BIAS_U:+.4f}m")
    
    stats = {
        'segment': segment_name,
        'solutions_count': len(df_analysis),
        'fix_count': len(df_analysis) if ONLY_FIXED_SOLUTIONS else (df_analysis['quality'] == 1).sum(),
        'fix_percentage': 100.0 if ONLY_FIXED_SOLUTIONS else (df_analysis['quality'] == 1).sum() / len(df_analysis) * 100,
        'mean_ns': df_analysis['ns'].mean() if 'ns' in df_analysis.columns else 0,
    }
    
    # Добавляем информацию о высотных ошибках (после смещения)
    if 'U' in df_analysis.columns:
        height_errors = df_analysis['U'].abs()
        stats['height_error_max'] = height_errors.max()
        stats['height_error_mean'] = height_errors.mean()
        stats['height_error_std'] = height_errors.std()
        stats['height_error_median'] = height_errors.median()
    
    # Добавляем информацию о 3D ошибках если есть
    if 'distance_3d' in df_analysis.columns:
        stats['3d_error_mean'] = df_analysis['distance_3d'].mean()
        stats['3d_error_std'] = df_analysis['distance_3d'].std()
        stats['3d_error_min'] = df_analysis['distance_3d'].min()
        stats['3d_error_max'] = df_analysis['distance_3d'].max()
        stats['3d_error_median'] = df_analysis['distance_3d'].median()
    
    # Статистика по ENU координатам (уже с поправкой)
    for component in ['E', 'N', 'U']:
        if component in df_analysis.columns:
            values = df_analysis[component].dropna()
            if len(values) > 0:
                # Основные статистические параметры
                stats[f'{component}_mean'] = values.mean()  # Среднее (систематическая ошибка ПОСЛЕ поправки)
                stats[f'{component}_std'] = values.std()    # СКО (случайная ошибка)
                stats[f'{component}_rms'] = np.sqrt(np.mean(values**2))  # СКП
                stats[f'{component}_min'] = values.min()
                stats[f'{component}_max'] = values.max()
                stats[f'{component}_median'] = values.median()
                stats[f'{component}_mad'] = (values - values.median()).abs().median()  # Median Absolute Deviation
    
    # 2D и 3D ошибки (после смещения)
    if all(col in df_analysis.columns for col in ['E', 'N']):
        E_vals = df_analysis['E'].dropna()
        N_vals = df_analysis['N'].dropna()
        if len(E_vals) > 0 and len(N_vals) > 0:
            horiz_errors = np.sqrt(E_vals**2 + N_vals**2)
            stats['2D_mean'] = horiz_errors.mean()
            stats['2D_std'] = horiz_errors.std()
            stats['2D_rms'] = np.sqrt(np.mean(horiz_errors**2))
            stats['2D_median'] = horiz_errors.median()
    
    if all(col in df_analysis.columns for col in ['E', 'N', 'U']):
        E_vals = df_analysis['E'].dropna()
        N_vals = df_analysis['N'].dropna()
        U_vals = df_analysis['U'].dropna()
        if len(E_vals) > 0 and len(N_vals) > 0 and len(U_vals) > 0:
            total_errors = np.sqrt(E_vals**2 + N_vals**2 + U_vals**2)
            stats['3D_mean'] = total_errors.mean()
            stats['3D_std'] = total_errors.std()
            stats['3D_rms'] = np.sqrt(np.mean(total_errors**2))
            stats['3D_median'] = total_errors.median()
    
    # Дополнительные метрики точности
    if all(col in stats for col in ['E_std', 'N_std', 'U_std']):
        stats['horizontal_precision'] = np.sqrt(stats['E_std']**2 + stats['N_std']**2)
        stats['vertical_precision'] = stats['U_std']
    
    return stats

def improved_main_analysis():
    """Улучшенная основная функция анализа с фильтрацией по ENU пределам"""
    
    # Загрузка всех решений
    print("📁 Загрузка POS-файлов...")
    solutions = {}
    pos_files = sorted(glob.glob(os.path.join(SOLUTIONS_DIR, "*.pos")))
    
    for file_path in pos_files:
        file_name = os.path.basename(file_path)
        df = read_pos_file_corrected(file_path)
        if len(df) > 0:
            solutions[file_name] = df
    
    if not solutions:
        print("❌ Не найдено POS-файлов для анализа")
        return None, None
    
    # Обработка каждого сегмента с фильтрацией
    solutions_stats = []
    processed_data = {}
    file_statistics = []
    
    print(f"\n🔄 Обработка {len(solutions)} сегментов с фильтрацией...")
    print(f"   Фильтры: {'ТОЛЬКО ФИКС' if ONLY_FIXED_SOLUTIONS else 'Все решения'}")
    print(f"   Пределы: E≤{MAX_EAST_ERROR}м, N≤{MAX_NORTH_ERROR}м, U≤{MAX_HEIGHT_ERROR}м")
    print(f"            2D≤{MAX_HORIZONTAL_ERROR}м, 3D≤{MAX_3D_ERROR}м")
    
    for file_name, df in solutions.items():
        print(f"\n🔍 Обработка {file_name}...")
        
        # Преобразование в ENU
        df_enu = reduction_to_enu(df.copy(), REF_POINT)
        
        if len(df_enu) > 0:
            # Статистика по файлу до фильтрации
            total_solutions = len(df_enu)
            fixed_solutions = len(df_enu[df_enu['quality'] == 1])
            
            file_statistics.append({
                'file_name': file_name,
                'total_solutions': total_solutions,
                'fixed_solutions': fixed_solutions,
                'fix_percentage': (fixed_solutions / total_solutions * 100) if total_solutions > 0 else 0
            })
            
            print(f"   Всего решений: {total_solutions}")
            print(f"   Фиксированных: {fixed_solutions} ({(fixed_solutions/total_solutions*100):.1f}%)")
            
            # Выбираем лучшие решения с учетом всех фильтров
            df_best = select_best_solutions(df_enu, SOL)
            
            if len(df_best) > 0:
                processed_data[file_name] = df_best
                
                # Расчет статистики
                stats = calculate_comprehensive_statistics(df_best, file_name)
                
                if stats:
                    solutions_stats.append(stats)
                    print(f"   ✅ Использовано {len(df_best)} решений")
                    print(f"   📊 Статистика:")
                    print(f"   ├─ СКО: E={stats.get('E_std', 0):.3f}m, N={stats.get('N_std', 0):.3f}m, U={stats.get('U_std', 0):.3f}m")
                    print(f"   ├─ Ошибка высоты: max={stats.get('height_error_max', 0):.3f}m")
                    if '3d_error_mean' in stats:
                        print(f"   └─ 3D ошибка: {stats['3d_error_mean']:.3f} ± {stats['3d_error_std']:.3f} м")
            else:
                print(f"   ⚠️  Нет решений, удовлетворяющих критериям")
        else:
            print(f"   ⚠️  Нет данных после преобразования в ENU")
    
    # Сводный анализ
    print(f"\n{'='*80}")
    print("РЕЗЮМЕ АНАЛИЗА С ФИЛЬТРАЦИЕЙ ПО ENU")
    print(f"{'='*80}")
    
    # Вывод информации о примененных фильтрах
    print(f"📋 ПАРАМЕТРЫ ФИЛЬТРАЦИИ:")
    print(f"   Тип решений: {'ТОЛЬКО ФИКСИРОВАННЫЕ' if ONLY_FIXED_SOLUTIONS else 'ВСЕ РЕШЕНИЯ'}")
    print(f"   Пределы ошибок:")
    print(f"     Восток (E): ≤ {MAX_EAST_ERROR} м")
    print(f"     Север (N): ≤ {MAX_NORTH_ERROR} м") 
    print(f"     Высота (U): ≤ {MAX_HEIGHT_ERROR} м")
    print(f"     Горизонталь (2D): ≤ {MAX_HORIZONTAL_ERROR} м")
    print(f"     3D: ≤ {MAX_3D_ERROR} м")
    print(f"   Количество решений на файл: {SOL}")
    
    if solutions_stats:
        df_summary = pd.DataFrame(solutions_stats)
        
        print(f"\n✅ АНАЛИЗИРУЕМЫЕ ДАННЫЕ:")
        print(f"   Файлов с подходящими решениями: {len(df_summary)}")
        print(f"   Всего проанализированных решений: {df_summary['solutions_count'].sum()}")
        
        print(f"\n{'='*80}")
        print(f"СВОДНЫЙ АНАЛИЗ ТОЧНОСТИ")
        print(f"{'='*80}")
        
        # Общая статистика
        print(f"\n📊 ОБЩАЯ СТАТИСТИКА ({len(df_summary)} файлов, {df_summary['solutions_count'].sum()} решений):")
        print_metrics_summary(df_summary)
        
        # Сохранение результатов
        # timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        # solution_type = "FIXED" if ONLY_FIXED_SOLUTIONS else "ALL"
        # output_file = f"ppp_analysis_{solution_type}_ENU_filtered_{timestamp}.csv"
        # df_summary.to_csv(output_file, index=False, encoding='utf-8-sig')
        # print(f"\n💾 Отчет сохранен в {output_file}")
        
        return processed_data, solutions_stats
    
    else:
        print(f"\n⚠️  Нет решений, соответствующих заданным критериям")
        return None, None

def print_metrics_summary(df):
    """Вывод сводки метрик для DataFrame"""
    
    sample_sizes = df['solutions_count']
    
    print(f"   📏 Размер выборки: {sample_sizes.mean():.1f} ± {sample_sizes.std():.1f} решений на файл")
    print(f"   ✅ Фикс: {df['fix_percentage'].mean():.1f} ± {df['fix_percentage'].std():.1f}%")
    
    metric_groups = [
        ("СКО", [
            ('E_std', 'E (м)'),
            ('N_std', 'N (м)'), 
            ('U_std', 'U (м)'),
        ]),
        ("СРЕДНЕЕ", [
            ('E_mean', 'E (м)'),
            ('N_mean', 'N (м)'),
            ('U_mean', 'U (м)'),
        ]),
        ("СКП", [
            ('E_rms', 'E (м)'),
            ('N_rms', 'N (м)'),
            ('U_rms', 'U (м)'),
        ])
    ]
    
    for group_name, metrics in metric_groups:
        print(f"   📈 {group_name}:")
        for col, desc in metrics:
            if col in df.columns:
                values = df[col].dropna()
                if len(values) > 0:
                    mean_val = values.mean()
                    std_val = values.std()
                    print(f"     {desc:8} {mean_val:7.4f} ± {std_val:6.4f} м")
    
    # 2D и 3D метрики
    if '2D_rms' in df.columns:
        print(f"   🎯 2D СКП: {df['2D_rms'].mean():.4f} ± {df['2D_rms'].std():.4f} м")
    if '3D_rms' in df.columns:
        print(f"   🎯 3D СКП: {df['3D_rms'].mean():.4f} ± {df['3D_rms'].std():.4f} м")
    
    # Статистика высоты
    if 'height_error_max' in df.columns:
        print(f"   📊 Ошибка высоты: max={df['height_error_max'].max():.4f} м, mean={df['height_error_mean'].mean():.4f} м")

# Запуск улучшенного анализа
if __name__ == "__main__":
    processed_data, statistics = improved_main_analysis()

📁 Загрузка POS-файлов...
✅ Загружено: 3411 записей (фикс: 0, 0.0%)
✅ Загружено: 5188 записей (фикс: 0, 0.0%)
✅ Загружено: 4977 записей (фикс: 0, 0.0%)
✅ Загружено: 4442 записей (фикс: 0, 0.0%)
✅ Загружено: 32 записей (фикс: 0, 0.0%)
✅ Загружено: 860 записей (фикс: 0, 0.0%)
✅ Загружено: 2575 записей (фикс: 0, 0.0%)
✅ Загружено: 28 записей (фикс: 0, 0.0%)
✅ Загружено: 0 записей (фикс: 0, 0.0%)
✅ Загружено: 0 записей (фикс: 0, 0.0%)

🔄 Обработка 8 сегментов с фильтрацией...
   Фильтры: Все решения
   Пределы: E≤0.2м, N≤0.2м, U≤0.3м
            2D≤15м, 3D≤20.0м

🔍 Обработка sol_000.pos...
   Всего решений: 3411
   Фиксированных: 0 (0.0%)
   Фильтры: E≤0.2м(-3292), N≤0.2м(-98)
   Итог: 3411 → 21 решений
   🚫 Выбросы: E-IQR(-3), N-IQR(-3), U-IQR(-4)
   Итог: 21 → 17 решений после IQR
   ✅ Отобрано 10 всех решений
   ✅ Использовано 10 решений
   📊 Статистика:
   ├─ СКО: E=0.016m, N=0.006m, U=0.008m
   ├─ Ошибка высоты: max=0.053m
   └─ 3D ошибка: 0.214 ± 0.001 м

🔍 Обработка sol_001.pos...
  